# Data Preprocessing — v1.0

## Summary
Cleaned and merged the constructor finance and F1 business datasets, resolved a structural missing-data issue, checked for target leakage, and produced two ready-to-use modeling datasets for the team.

## What's included

**Missing values**
- `cost_cap_limit_usd_m` had 104 missing values, all in pre-2021 rows (`in_cost_cap_era == 0`). Confirmed this is structural — no cost cap existed before 2021, so there was nothing to record — not a data quality issue. Filled with `0` as a sentinel, valid only in combination with `in_cost_cap_era`. Verified zero missing values in cap-era rows.
- No other missing values found across the remaining four source files.

**Merge**
- Joined `constructor_finances.csv` with `f1_business.csv` on `season` (left join). Row count confirmed unchanged at 164 — no duplication from the join.

**Leakage check**
- `total_revenue_usd_m` (team-level) is arithmetically derived from `sponsorship_revenue_usd_m + prize_money_usd_m` for the same row — excluded from all model features, since it's derived directly from our regression target.
- `total_revenue_usd_m_group` (from `f1_business.csv`) is a season-wide F1 business total, independent of any single team. Verified via `groupby(season).nunique() == 1` across all 17 seasons. Kept as a feature.

**Encoding**
- `in_cost_cap_era` and `is_constructors_champion` converted to integer flags.
- `team_name` one-hot encoded (`drop_first=True`).

**Outputs**
- `regression_dataset.csv` — target: `prize_money_usd_m`
- `classification_dataset.csv` — target: `is_constructors_champion`
- Both share the same feature set; both targets are excluded from each other's feature columns to keep the two tasks independent.
- 80/20 train/test split for both. Classification split is stratified on the target to preserve the ~10% champion ratio, given the class imbalance (17 champion-seasons out of 164 rows).

## Notes
- `total_revenue_usd_m` is still present in the full merged dataframe for EDA/statistical analysis — it's only excluded from the modeling feature set.
- `sponsorship_revenue_usd_m` and `prize_money_usd_m` remain in the feature set even though they combine to form the excluded column — this is a deliberate choice, not an oversight, since they're independent inputs rather than derived from the target.
- Full reasoning for both decisions above is documented here so you guys don't need to re-derive it.

In [ ]:
import os

import kagglehub
import pandas as pd


path = kagglehub.dataset_download("sergionefedov/formula-1-racing-industry-2010-2026")

print("Path to dataset files:", path)
list_directory = os.listdir(path)
print("List of files:", list_directory)

#extracting each table from the dataset
circuit_economics_data = pd.read_csv(os.path.join(path, "circuit_economics.csv"))
constructor_finances_data = pd.read_csv(os.path.join(path, "constructor_finances.csv"))
driver_standings_data = pd.read_csv(os.path.join(path, "driver_standings.csv"))
f1_business_data = pd.read_csv(os.path.join(path, "f1_business.csv"))
race_results_data = pd.read_csv(os.path.join(path, "race_results.csv"))  

data_list = [circuit_economics_data, constructor_finances_data, driver_standings_data, f1_business_data, race_results_data]
for df in data_list:
    print(df.head()) 

Path to dataset files: C:\Users\kofia\.cache\kagglehub\datasets\sergionefedov\formula-1-racing-industry-2010-2026\versions\1
List of files: ['circuit_economics.csv', 'constructor_finances.csv', 'driver_standings.csv', 'f1_business.csv', 'race_results.csv']
   season grand_prix circuit country_iso3       region  hosting_fee_usd_m_est  \
0    2010    Bahrain  Sakhir          BHR  Middle East                   38.5   
1    2012    Bahrain  Sakhir          BHR  Middle East                   35.8   
2    2013    Bahrain  Sakhir          BHR  Middle East                   34.6   
3    2014    Bahrain  Sakhir          BHR  Middle East                   35.1   
4    2015    Bahrain  Sakhir          BHR  Middle East                   34.2   

   weekend_attendance_k  debut_year  is_street_circuit  \
0                    97        2004                  0   
1                   100        2004                  0   
2                    94        2004                  0   
3                    96 

In [ ]:
#Of course, we need to  see the shape, dtypes, and check for missing values and duplicate rows.
#Our analysis needs a factual baseline. We need to know what we're working with.
for name, df in zip(
    ["circuit_economics", "constructor_finances", "driver_standings", "f1_business", "race_results"],
    [circuit_economics_data, constructor_finances_data, driver_standings_data, f1_business_data, race_results_data]
):
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("dtypes:\n", df.dtypes)
    print("missing values:\n", df.isna().sum()[df.isna().sum() > 0])
    print("duplicate rows:", df.duplicated().sum())
    
    #Yes, there are missing values, but upon further investigation, it's not a cause for concern.
    #It's the cost_cap_limit_usd_m column, in the missing values are from when this was not yet introduced (it only started in 2021)
    



--- circuit_economics ---
shape: (277, 10)
dtypes:
 season                        int64
grand_prix                      str
circuit                         str
country_iso3                    str
region                          str
hosting_fee_usd_m_est       float64
weekend_attendance_k          int64
debut_year                    int64
is_street_circuit             int64
is_new_circuit_post_2020      int64
dtype: object
missing values:
 Series([], dtype: int64)
duplicate rows: 0

--- constructor_finances ---
shape: (164, 10)
dtypes:
 season                         int64
team_name                        str
country_iso3                     str
operating_budget_usd_m       float64
sponsorship_revenue_usd_m    float64
prize_money_usd_m            float64
total_revenue_usd_m          float64
in_cost_cap_era                int64
cost_cap_limit_usd_m         float64
is_constructors_champion       int64
dtype: object
missing values:
 cost_cap_limit_usd_m    104
dtype: int64
duplicate rows:

In [ ]:
#make a copy of constructor_finances_data
cf = constructor_finances_data.copy()

#In this step, we replace the missing values in the cost_cap_limit_usd_m column with 0.
cf["cost_cap_limit_usd_m"] = cf["cost_cap_limit_usd_m"].fillna(0)


#In this step, we do an outlier check on the money columns, using IQR flags.
#Anything below Q1 - 1.5xIQR or above Q3 + 1.5xIQR is flagged. On the four money(continuous) columns.
money_cols = ["operating_budget_usd_m", "sponsorship_revenue_usd_m",
              "prize_money_usd_m", "total_revenue_usd_m"]

for col in money_cols:
    q1, q3 = cf[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = cf[(cf[col] < lo) | (cf[col] > hi)]
    print(f"{col}: {len(flagged)} outlier rows (bounds {lo:.1f}–{hi:.1f})")
    if len(flagged):
        print(flagged[["season", "team_name", col]])

operating_budget_usd_m: 0 outlier rows (bounds -171.6–631.9)
sponsorship_revenue_usd_m: 0 outlier rows (bounds -124.6–404.8)
prize_money_usd_m: 0 outlier rows (bounds -78.7–296.9)
total_revenue_usd_m: 0 outlier rows (bounds -206.9–718.8)


In [ ]:
#Here, we need to join the relevant columns from the separate tables derived from the dataset.
#This is typically a left join on season, attaching the F1-group-wide business context to each team-season row.
#The how="left" keeps all team-season rows
merged = cf.merge(f1_business_data, on="season", how="left", suffixes=("", "_group"))
print(merged.shape)

(164, 19)


In [ ]:
#Encoding categorical variables
#Converting the boolean columns(is_cost_cap_era, is_constructors_champion) to actual 0/1 integers, and one-hot encodes team_name into separate_binary columns.
#Doing this for the sake of efficiency and correctness. The model can't read the word "Mercedes." It can only read 0/1 integers. 
merged["in_cost_cap_era"] = merged["in_cost_cap_era"].astype(int)
merged["is_constructors_champion"] = merged["is_constructors_champion"].astype(int)

# team_name: one-hot for linear models, but we keep the raw column too 
#drop_first=True: just to drop one team column to avoid redundancy
merged = pd.get_dummies(merged, columns=["team_name"], prefix="team", drop_first=True)

merged

,season,country_iso3,operating_budget_usd_m,sponsorship_revenue_usd_m,prize_money_usd_m,total_revenue_usd_m,in_cost_cap_era,cost_cap_limit_usd_m,is_constructors_champion,total_revenue_usd_m_group,...,global_avg_race_viewers_m,team_Aston Martin,team_Ferrari,team_Haas,team_McLaren,team_Mercedes,team_RB (AlphaTauri),team_Red Bull Racing,team_Stake (Sauber),team_Williams
0,2010,DEU,270.0,86.2,112.6,198.8,0,0.0,0,1108.0,...,70.0,False,False,False,False,True,False,False,False,False
1,2011,DEU,320.5,100.2,118.4,218.6,0,0.0,0,1265.0,...,71.5,False,False,False,False,True,False,False,False,False
2,2012,DEU,308.3,102.4,130.4,232.9,0,0.0,0,1377.0,...,73.0,False,False,False,False,True,False,False,False,False
3,2013,DEU,356.9,126.9,158.6,285.6,0,0.0,0,1509.0,...,74.5,False,False,False,False,True,False,False,False,False
4,2014,DEU,339.4,128.8,173.6,302.3,0,0.0,1,1728.0,...,76.0,False,False,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159,2022,USA,125.7,86.5,51.6,138.1,1,140.0,0,2486.0,...,88.0,False,False,True,False,False,False,False,False,False
160,2023,USA,140.7,103.4,53.8,157.3,1,135.0,0,3208.0,...,89.5,False,False,True,False,False,False,False,False,False
161,2024,USA,147.2,118.8,62.6,181.4,1,135.0,0,3782.0,...,91.0,False,False,True,False,False,False,False,False,False
162,2025,USA,150.5,131.6,80.1,211.8,1,135.0,0,3933.0,...,92.5,False,False,True,False,False,False,False,False,False


In [ ]:
# total_revenue_usd_m = sponsorship_revenue_usd_m + prize_money_usd_m (same row)
# Keep it for descriptive stats/plots, but exclude from BOTH model feature sets.
#Flagging it as possible leakage data.
leak_cols = ["total_revenue_usd_m"]

#confirming the uniqueness of the total revenue per season
merged.groupby("season")["total_revenue_usd_m_group"].nunique()


season
2010    1
2011    1
2012    1
2013    1
2014    1
2015    1
2016    1
2017    1
2018    1
2019    1
2020    1
2021    1
2022    1
2023    1
2024    1
2025    1
2026    1
Name: total_revenue_usd_m_group, dtype: int64

In [ ]:
#we drop the leaked columns at the point of feature selection
feature_cols = [c for c in merged.columns if c not in leak_cols + ["prize_money_usd_m", "is_constructors_champion"]]

# regression
regression_df = merged[feature_cols + ["prize_money_usd_m"]]

# classification
classification_df = merged[feature_cols + ["is_constructors_champion"]]

#so we build two separate CSVs -- one with prize_money_usd_m as the target, and one with is_constructors_champion as the target (regression and classification respectively)
regression_df.to_csv("data/regression_dataset.csv", index=False)
classification_df.to_csv("data/classification_dataset.csv", index=False)

In [ ]:
from sklearn.model_selection import train_test_split

# Regression: a simple random split is fine here
X_reg = regression_df.drop(columns=["prize_money_usd_m"])
y_reg = regression_df["prize_money_usd_m"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Classification: stratify on the target — only ~17 positive cases out of 164, so a random
# split could easily put zero champions in the test set otherwise
X_clf = classification_df.drop(columns=["is_constructors_champion"])
y_clf = classification_df["is_constructors_champion"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)